# FinQA teacher scoring on Colab GPU

This notebook clones tested code into Colab's fast local disk and stores only expensive artifacts in Google Drive. It runs development baselines and Qwen teacher-score caching, and never evaluates the test split.

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import os
import subprocess

drive.mount('/content/drive')
REPOSITORY = 'WatermelonfromEarth/finevid-distill'
PROJECT_DIR = Path('/content/finevid-distill')
ARTIFACT_ROOT = Path('/content/drive/MyDrive/FinEvid-Distill')
TEACHER_DIR = ARTIFACT_ROOT / 'teacher_scores'
CHECKPOINT_DIR = ARTIFACT_ROOT / 'checkpoints'
RESULTS_DIR = ARTIFACT_ROOT / 'experiment_results'
for directory in (TEACHER_DIR, CHECKPOINT_DIR, RESULTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# Add a read-only GITHUB_TOKEN under Colab's key icon because the repository is private.
token = userdata.get('GITHUB_TOKEN')
assert token, 'Add a read-only GITHUB_TOKEN in Colab Secrets before continuing.'
repository_url = f'https://github.com/{REPOSITORY}.git'
askpass = Path('/content/finevid-git-askpass.sh')
askpass.write_text("#!/bin/sh\ncase \"$1\" in\n  *Username*) printf '%s\\n' 'x-access-token' ;;\n  *) printf '%s\\n' \"$GITHUB_TOKEN\" ;;\nesac\n")
askpass.chmod(0o700)
git_environment = os.environ.copy()
git_environment.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': token})
def run_git(*arguments):
    result = subprocess.run(['git', *arguments], env=git_environment, capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError((result.stderr or result.stdout).replace(token, '[REDACTED]'))
if PROJECT_DIR.exists():
    run_git('-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', 'main')
else:
    run_git('clone', '--depth', '1', repository_url, str(PROJECT_DIR))
run_git('-C', str(PROJECT_DIR), 'remote', 'set-url', 'origin', repository_url)
askpass.unlink(missing_ok=True)
del token, git_environment
print('Code:', PROJECT_DIR)
print('Persistent artifacts:', ARTIFACT_ROOT)

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU before continuing.'
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)

## Install the project without replacing Colab's CUDA-enabled PyTorch

In [ ]:
%pip install -q -r {PROJECT_DIR / 'requirements-colab.txt'}
%pip install -q -e {PROJECT_DIR} --no-deps

In [ ]:
import importlib
import sys

source_directory = str(PROJECT_DIR / 'src')
sys.path_importer_cache.pop(source_directory, None)
sys.path.insert(0, source_directory)
importlib.invalidate_caches()

def run_project(*arguments):
    return subprocess.run([sys.executable, *arguments], cwd=PROJECT_DIR, check=True)

run_project('-m', 'pytest', 'tests/test_metrics.py', 'tests/test_baselines_and_teacher_cache.py', '-q')

## Reproduce the three non-trained development baselines

BGE uses the pinned model revision, the official query instruction, plain candidate text, normalized embeddings, and cosine similarity.

In [ ]:
run_project(
    'src/evaluation/evaluate.py',
    '--models', 'random,bm25,bge',
    '--device', 'cuda',
    '--batch-size', '256',
    '--output', str(RESULTS_DIR / 'dev_baselines.json'),
)

## Cache every development teacher score

Scores are raw Qwen yes-minus-no logit differences. No sigmoid, softmax, or distillation temperature is applied. The `.partial` file resumes at the next whole question after a disconnect.

In [ ]:
run_project(
    'src/data/cache_teacher_scores.py',
    '--splits', 'dev',
    '--device', 'cuda',
    '--batch-size', '16',
    '--pair-chunk-size', '2048',
    '--output-dir', str(TEACHER_DIR),
)

## Apply the teacher quality gate

This runs all systems through the same metric functions. Training-score caching is blocked unless teacher development MRR exceeds frozen BGE development MRR.

In [ ]:
run_project(
    'src/evaluation/evaluate.py',
    '--models', 'random,bm25,bge,teacher',
    '--device', 'cuda',
    '--batch-size', '256',
    '--teacher-cache', str(TEACHER_DIR / 'teacher_dev_scores.jsonl'),
    '--output', str(RESULTS_DIR / 'dev_teacher_comparison.json'),
)

In [ ]:
import json

comparison = json.loads((RESULTS_DIR / 'dev_teacher_comparison.json').read_text())
frozen_bge_mrr = comparison['models']['Frozen BGE']['mrr']
teacher_mrr = comparison['models']['Qwen teacher']['mrr']
teacher_gate_passed = teacher_mrr > frozen_bge_mrr
print(f'Frozen BGE MRR: {frozen_bge_mrr:.6f}')
print(f'Qwen teacher MRR: {teacher_mrr:.6f}')
assert teacher_gate_passed, (
    'Teacher gate failed. Stop before training and inspect the financial retrieval instruction, '
    'table serialization, and token truncation.'
)
print('PASS: Qwen teacher outperforms frozen BGE on development.')

## Cache training logits only after the gate passes

This is the long-running cell. Its output is written directly to Drive and can resume safely.

In [ ]:
assert teacher_gate_passed
run_project(
    'src/data/cache_teacher_scores.py',
    '--splits', 'train',
    '--device', 'cuda',
    '--batch-size', '16',
    '--pair-chunk-size', '2048',
    '--output-dir', str(TEACHER_DIR),
)

In [ ]:
from finevid_distill.data.cache_teacher_scores import validate_teacher_cache
from finevid_distill.evaluation.evaluate import load_jsonl

for split in ('dev', 'train'):
    records = load_jsonl(PROJECT_DIR / f'data/processed/{split}.jsonl')
    cache = TEACHER_DIR / f'teacher_{split}_scores.jsonl'
    print(split, validate_teacher_cache(cache, records))